# Task 3, Part A: Images as Tensors

Welcome! In this notebook we'll explore how **PyTorch** stores images as
tensors.  By the end you'll understand:

* The `(C, H, W)` convention PyTorch uses for images.
* How to load and visualise the **MNIST** and **CIFAR-10** datasets.
* Basic pixel-level statistics of image tensors.
* How a `DataLoader` batches images for training.

> **Pre-requisites:** Basic Python.  Everything else is explained as we go!

## 1 · Setup and MNIST

We need two libraries:
| Library | Purpose |
|---|---|
| `torch` | Core tensor operations |
| `torchvision` | Ready-made datasets & image transforms |

Let's install them (skip if already installed) and import what we need.

In [ ]:
# Install torchvision (and torch) if not already present
# The '%%capture' magic silences the noisy pip output.
%%capture
!pip install torch torchvision

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

print(f"PyTorch version : {torch.__version__}")

### Download MNIST

MNIST is a classic dataset of **28 × 28 grayscale** handwritten digits (0-9).

We apply `transforms.ToTensor()` so every image is converted to a
`torch.Tensor` with pixel values in `[0, 1]`.

In [ ]:
# Download and load the MNIST training set
mnist_train = datasets.MNIST(
    root='./data',          # folder to save data in
    train=True,             # training split
    download=True,          # download if not already present
    transform=transforms.ToTensor()  # PIL Image → Tensor
)

# Grab a single image and its label
image, label = mnist_train[0]

print(f"Type  : {type(image)}")
print(f"Shape : {image.shape}   ← (C, H, W)")
print(f"Label : {label}")
print()
print("PyTorch stores images in (C, H, W) order:")
print("  C = number of channels  (1 for grayscale, 3 for RGB)")
print("  H = height in pixels")
print("  W = width  in pixels")
print()
print("For MNIST, C = 1 because the images are grayscale.")

## 2 · Visualising MNIST

Let's look at the first **16** images in a 4 × 4 grid.

Because each image has shape `(1, 28, 28)` and `plt.imshow` expects
`(H, W)` for grayscale, we `.squeeze()` out the channel dimension.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(6, 6))

for idx, ax in enumerate(axes.flat):
    img, lbl = mnist_train[idx]
    # squeeze: (1, 28, 28) → (28, 28) so matplotlib can display it
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Label: {lbl}', fontsize=9)
    ax.axis('off')

plt.suptitle('First 16 MNIST images', fontsize=14)
plt.tight_layout()
plt.show()

## 3 · Pixel Statistics

Understanding the range and distribution of pixel values is important
before feeding data into a neural network.

In [ ]:
# We'll compute stats over the very first image
image, _ = mnist_train[0]

print(f"Min pixel value  : {image.min().item():.4f}")
print(f"Max pixel value  : {image.max().item():.4f}")
print(f"Mean pixel value : {image.mean().item():.4f}")
print()
print("Because we used ToTensor(), values are scaled to [0, 1].")
print("0 = black, 1 = white.")

## 4 · CIFAR-10 — Colour Images

CIFAR-10 contains **32 × 32 RGB** images across 10 classes:

| Index | Class |
|-------|-------|
| 0 | airplane |
| 1 | automobile |
| 2 | bird |
| 3 | cat |
| 4 | deer |
| 5 | dog |
| 6 | frog |
| 7 | horse |
| 8 | ship |
| 9 | truck |

In [ ]:
# Download CIFAR-10
cifar_train = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

# Inspect one image
image, label = cifar_train[0]
print(f"Shape : {image.shape}   ← (C, H, W)")
print(f"Label : {label} ({cifar_train.classes[label]})")
print()
print("C = 3 here because CIFAR-10 images are RGB (Red, Green, Blue).")

### Visualise 16 CIFAR-10 images

For colour images `plt.imshow` expects shape `(H, W, C)`, so we
need to **permute** the tensor from `(C, H, W)` → `(H, W, C)`.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(7, 7))

for idx, ax in enumerate(axes.flat):
    img, lbl = cifar_train[idx]
    # permute: (C, H, W) → (H, W, C) for matplotlib
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(cifar_train.classes[lbl], fontsize=9)
    ax.axis('off')

plt.suptitle('First 16 CIFAR-10 images', fontsize=14)
plt.tight_layout()
plt.show()

## 5 · DataLoader — Batching Images

In practice we don't feed images one-by-one.  A `DataLoader` groups
them into **batches** so training is faster and more stable.

With `batch_size=64`, each batch will contain 64 images stacked into a
single tensor of shape `(64, C, H, W)`.

In [ ]:
# Create a DataLoader for CIFAR-10
train_loader = DataLoader(
    cifar_train,
    batch_size=64,    # 64 images per batch
    shuffle=True      # randomise order each epoch
)

# Grab one batch
x_batch, y_batch = next(iter(train_loader))

print(f"x (images) shape : {x_batch.shape}")
print(f"y (labels) shape : {y_batch.shape}")
print()
print("x has 4 dimensions: (batch_size, C, H, W)")
print("y is a 1-D tensor of integer class labels, one per image.")

## Recap

| Concept | Key takeaway |
|---|---|
| **Image shape** | PyTorch uses `(C, H, W)` — channels first |
| **ToTensor()** | Converts a PIL image to a float tensor in `[0, 1]` |
| **Grayscale** | `C = 1` (e.g. MNIST) |
| **RGB** | `C = 3` (e.g. CIFAR-10) |
| **DataLoader** | Stacks images into batches of shape `(B, C, H, W)` |

Next up → **Part B**: applying transforms and building a simple CNN. 🚀